In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy scipy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Executar jobs em uma sessão

<details>
<summary><b>Versões dos pacotes</b></summary>

O código nesta página foi desenvolvido usando os seguintes requisitos.
Recomendamos usar estas versões ou mais recentes.

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
scipy~=1.16.3
```
</details>
> **Note:** Usuários do Open Plan não podem enviar jobs de sessão. As cargas de trabalho devem ser executadas no [modo job](/guides/execution-modes#job-mode) ou no [modo batch](/guides/execution-modes#batch-mode).

Use sessões quando você precisar de acesso dedicado e exclusivo ao QPU.

## Configurar para usar sessões
Antes de iniciar uma sessão, você deve [configurar o Qiskit Runtime](./install-qiskit) e inicializá-lo como serviço:

In [1]:
from qiskit_ibm_runtime import (
    QiskitRuntimeService,
    Session,
    SamplerV2 as Sampler,
    EstimatorV2 as Estimator,
)

service = QiskitRuntimeService()

## Abrir uma sessão
Você pode abrir uma sessão de runtime usando o gerenciador de contexto `with Session(...)` ou inicializando a classe `Session`.
Ao iniciar uma sessão, você deve especificar um QPU passando um objeto `backend`. A sessão começa quando seu primeiro job inicia a execução.

> **Note:** Se você abrir uma sessão, mas não enviar nenhum job a ela por 30 minutos, a sessão é fechada automaticamente.

**Classe Session**

> **Caution:** O bloco de código a seguir retornará um erro para usuários do Open Plan porque usa sessões. As cargas de trabalho no Open Plan só podem ser executadas no [modo job](/guides/execution-modes#job-mode) ou no [modo batch](/guides/execution-modes#batch-mode).

In [2]:
backend = service.least_busy(operational=True, simulator=False)
session = Session(backend=backend)
estimator = Estimator(mode=session)
sampler = Sampler(mode=session)
# Close the session because no context manager was used.
session.close()

**Gerenciador de contexto**

O gerenciador de contexto abre e fecha a sessão automaticamente.

> **Caution:** O bloco de código a seguir retornará um erro para usuários do Open Plan porque usa sessões. As cargas de trabalho no Open Plan só podem ser executadas no [modo job](/guides/execution-modes#job-mode) ou no [modo batch](/guides/execution-modes#batch-mode).

In [3]:
from qiskit_ibm_runtime import (
    Session,
    SamplerV2 as Sampler,
    EstimatorV2 as Estimator,
)

backend = service.least_busy(operational=True, simulator=False)
with Session(backend=backend):
    estimator = Estimator()
    sampler = Sampler()

<span id="specify-length"></span>
## Duração da sessão
O tempo de vida máximo da sessão (TTL) determina por quanto tempo uma sessão pode ser executada. Você pode definir esse valor com o parâmetro `max_time`. Esse valor deve ser superior ao tempo de execução do job mais longo.

Este temporizador começa quando a sessão inicia. Quando o valor é atingido, a sessão é encerrada. Os jobs que estiverem em execução serão concluídos, mas os jobs ainda na fila serão cancelados com falha.

> **Caution:** O bloco de código a seguir retornará um erro para usuários do Open Plan porque usa sessões. As cargas de trabalho no Open Plan só podem ser executadas no [modo job](/guides/execution-modes#job-mode) ou no [modo batch](/guides/execution-modes#batch-mode).

In [4]:
from qiskit.quantum_info import SparsePauliOp
from qiskit.circuit import QuantumCircuit, Parameter
from qiskit.transpiler import generate_preset_pass_manager
import numpy as np

# This cell is hidden from users
service = QiskitRuntimeService()
backend = service.least_busy()

# Define two circuits, each with one parameter with two parameters.
circuit = QuantumCircuit(2)
circuit.h(0)
circuit.cx(0, 1)
circuit.ry(Parameter("a"), 0)
circuit.cx(0, 1)
circuit.h(0)
circuit.measure_all()


pm = generate_preset_pass_manager(optimization_level=1, backend=backend)
transpiled_circuit = pm.run(circuit)
transpiled_circuit_sampler = transpiled_circuit
transpiled_circuit_sampler.measure_all()

# Create parameters and mapped observables to submit
params = np.random.uniform(size=(2, 3)).T
observables = [
    SparsePauliOp(["XX", "IY"], [0.5, 0.5]),
    SparsePauliOp("XX"),
    SparsePauliOp("IY"),
]
mapped_observables = [
    [observable.apply_layout(transpiled_circuit.layout)]
    for observable in observables
]

sampler_pub = (transpiled_circuit_sampler, params)
estimator_pub = (transpiled_circuit_sampler, mapped_observables, params)

In [5]:
with Session(backend=backend) as session:
    estimator = Estimator()
    sampler = Sampler()
    job1 = estimator.run([estimator_pub])
    job2 = sampler.run([sampler_pub])

# The session is no longer accepting jobs but the submitted job will run to completion.
result = job1.result()
result2 = job2.result()

<Admonition type="tip">
If you are not using a context manager, manually close the session to avoid unwanted cost. You can close a session as soon as you are done submitting jobs to it. When a session is closed with `session.close()`, it no longer accepts new jobs, but the already submitted jobs will still run until completion and their results can be retrieved.
</Admonition>


<Admonition type="caution">
The following code block will return an error for users on the Open Plan because it uses sessions. Workloads on the Open Plan can run only in [job mode](/docs/guides/execution-modes#job-mode) or [batch mode](/docs/guides/execution-modes#batch-mode).
</Admonition>

In [6]:
session = Session(backend=backend)

# If using qiskit-ibm-runtime earlier than 0.24.0, change `mode=` to `session=`
estimator = Estimator(mode=session)
sampler = Sampler(mode=session)
job1 = estimator.run([estimator_pub])
job2 = sampler.run([sampler_pub])
print(f"Result1: {job1.result()}")
print(f"Result2: {job2.result()}")

# Manually close the session. Running and queued jobs will run to completion.
session.close()

Result1: PrimitiveResult([PubResult(data=DataBin(evs=np.ndarray(<shape=(3, 2), dtype=float64>), stds=np.ndarray(<shape=(3, 2), dtype=float64>), ensemble_standard_error=np.ndarray(<shape=(3, 2), dtype=float64>), shape=(3, 2)), metadata={'shots': 4096, 'target_precision': 0.015625, 'circuit_metadata': {}, 'resilience': {}, 'num_randomizations': 32})], metadata={'dynamical_decoupling': {'enable': False, 'sequence_type': 'XX', 'extra_slack_distribution': 'middle', 'scheduling_method': 'alap'}, 'twirling': {'enable_gates': False, 'enable_measure': True, 'num_randomizations': 'auto', 'shots_per_randomization': 'auto', 'interleave_randomizations': True, 'strategy': 'active-accum'}, 'resilience': {'measure_mitigation': True, 'zne_mitigation': False, 'pec_mitigation': False}, 'version': 2})


Result2: PrimitiveResult([SamplerPubResult(data=DataBin(meas=BitArray(<shape=(3, 2), num_shots=4096, num_bits=2>), meas0=BitArray(<shape=(3, 2), num_shots=4096, num_bits=133>), shape=(3, 2)), metadata={'circuit_metadata': {}})], metadata={'execution': {'execution_spans': ExecutionSpans([DoubleSliceSpan(<start='2026-01-15 07:53:15', stop='2026-01-15 07:53:21', size=24576>)])}, 'version': 2})


> **Tip:** Se você não estiver usando um gerenciador de contexto, feche a sessão manualmente para evitar custos indesejados. Você pode fechar uma sessão assim que terminar de enviar jobs a ela. Quando uma sessão é fechada com `session.close()`, ela não aceita mais novos jobs, mas os jobs já enviados ainda serão executados até a conclusão e seus resultados podem ser recuperados.

> **Caution:** O bloco de código a seguir retornará um erro para usuários do Open Plan porque usa sessões. As cargas de trabalho no Open Plan só podem ser executadas no [modo job](/guides/execution-modes#job-mode) ou no [modo batch](/guides/execution-modes#batch-mode).

In [7]:
from qiskit_ibm_runtime import (
    QiskitRuntimeService,
    Session,
    EstimatorV2 as Estimator,
)

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

with Session(backend=backend) as session:
    print(session.details())

{'id': 'be84569d-86b5-4a7f-be5e-7d33e80dc220', 'backend_name': 'ibm_torino', 'interactive_timeout': 60, 'max_time': 28800, 'active_timeout': 28800, 'state': 'open', 'accepting_jobs': True, 'last_job_started': None, 'last_job_completed': None, 'started_at': None, 'closed_at': None, 'activated_at': None, 'mode': 'dedicated', 'usage_time': None}


## Usage patterns

Sessions are especially useful for algorithms that require frequent communication between classical and quantum resources.

Example: Run an iterative workload that uses the classical SciPy optimizer to minimize a cost function. In this model, SciPy uses the output of the cost function to calculate its next input.


<Admonition type="caution">
The following code block will return an error for users on the Open Plan because it uses sessions. Workloads on the Open Plan can run only in [job mode](/docs/guides/execution-modes#job-mode) or [batch mode](/docs/guides/execution-modes#batch-mode).
</Admonition>

In [8]:
from scipy.optimize import minimize
from qiskit.circuit.library import efficient_su2


def cost_func(params, ansatz, hamiltonian, estimator):
    # Return estimate of energy from estimator

    energy = sum(
        estimator.run([(ansatz, hamiltonian, params)]).result()[0].data.evs
    )
    return energy


hamiltonian = SparsePauliOp.from_list(
    [("YZ", 0.3980), ("ZI", -0.3980), ("ZZ", -0.0113), ("XX", 0.1810)]
)
su2_ansatz = efficient_su2(hamiltonian.num_qubits)
pm = generate_preset_pass_manager(backend=backend, optimization_level=3)
ansatz = pm.run(su2_ansatz)
mapped_hamiltonian = [
    operator.apply_layout(ansatz.layout) for operator in hamiltonian
]

num_params = ansatz.num_parameters
x0 = 2 * np.pi * np.random.random(num_params)

session = Session(backend=backend)


# If using qiskit-ibm-runtime earlier than 0.24.0, change `mode=` to `session=`
estimator = Estimator(mode=session, options={"default_shots": int(1e4)})
res = minimize(
    cost_func,
    x0,
    args=(ansatz, mapped_hamiltonian, estimator),
    method="cobyla",
    options={"maxiter": 25},
)

# Close the session because no context manager was used.
session.close()

<span id="two-vqe"></span>
## Run two VQE algorithms in a session by using threading

You can get more out of a session by running multiple workloads simultaneously. The following example shows how you can run two VQE algorithms, each using a different classical optimizer, simultaneously inside a single session. Job tags are also used to differentiate jobs from each workload.


<Admonition type="caution">
The following code block will return an error for users on the Open Plan because it uses sessions. Workloads on the Open Plan can run only in [job mode](/docs/guides/execution-modes#job-mode) or [batch mode](/docs/guides/execution-modes#batch-mode).
</Admonition>

In [9]:
from concurrent.futures import ThreadPoolExecutor
from qiskit_ibm_runtime import EstimatorV2 as Estimator


def minimize_thread(estimator, method):
    return minimize(
        cost_func,
        x0,
        args=(ansatz, mapped_hamiltonian, estimator),
        method=method,
        options={"maxiter": 25},
    )


with Session(backend=backend), ThreadPoolExecutor() as executor:
    estimator1 = Estimator()
    estimator2 = Estimator()

    # Use different tags to differentiate the jobs.
    estimator1.options.environment.job_tags = ["cobyla"]
    estimator2.options.environment.job_tags = ["nelder-mead"]

    # Submit the two workloads.
    cobyla_future = executor.submit(minimize_thread, estimator1, "cobyla")
    nelder_mead_future = executor.submit(
        minimize_thread, estimator2, "nelder-mead"
    )

    # Get workload results.
    cobyla_result = cobyla_future.result()
    nelder_mead_result = nelder_mead_future.result()

<span id="session-status"></span>
## Verificar o status da sessão
Você pode consultar o status de uma sessão para entender seu estado atual usando `session.status()` ou acessando a página [Workloads](https://quantum.cloud.ibm.com/workloads).

O status da sessão pode ser um dos seguintes:

- `Pending`: A sessão não foi iniciada ou foi desativada. O próximo job de sessão precisa aguardar na fila como outros jobs.
- `In progress, accepting new jobs`: A sessão está ativa e aceitando novos jobs.
- `In progress, not accepting new jobs`: A sessão está ativa, mas não está aceitando novos jobs. O envio de jobs à sessão é rejeitado, mas os jobs de sessão pendentes serão executados até a conclusão. A sessão é fechada automaticamente quando todos os jobs terminarem.
- `Closed`: O valor máximo de timeout da sessão foi atingido ou a sessão foi explicitamente fechada.

<span id="session-details"></span>
## Determinar os detalhes da sessão
Para uma visão geral completa da configuração e do status de uma sessão, use o método `session.details()`.

> **Caution:** O bloco de código a seguir retornará um erro para usuários do Open Plan porque usa sessões. As cargas de trabalho no Open Plan só podem ser executadas no [modo job](/guides/execution-modes#job-mode) ou no [modo batch](/guides/execution-modes#batch-mode).